In [2]:
import os
import random
import re
import sys

DAMPING = 0.85
SAMPLES = 10000


def main():
    if len(sys.argv) != 2:
        sys.exit("Usage: python pagerank.py corpus")
    corpus = crawl(sys.argv[1])
    ranks = sample_pagerank(corpus, DAMPING, SAMPLES)
    print(f"PageRank Results from Sampling (n = {SAMPLES})")
    for page in sorted(ranks):
        print(f"  {page}: {ranks[page]:.4f}")
    ranks = iterate_pagerank(corpus, DAMPING)
    print("PageRank Results from Iteration")
    for page in sorted(ranks):
        print(f"  {page}: {ranks[page]:.4f}")


def crawl(directory):
    """
    Parse a directory of HTML pages and check for links to other pages.
    Return a dictionary where each key is a page, and values are
    a list of all other pages in the corpus that are linked to by the page.
    """
    pages = dict()

    for filename in os.listdir(directory):
        if not filename.endswith(".html"):
            continue
        with open(os.path.join(directory, filename)) as f:
            contents = f.read()
            links = re.findall(r"<a\s+(?:[^>]*?)href=\"([^\"]*)\"", contents)
            pages[filename] = set(links) - {filename}

    for filename in pages:
        pages[filename] = set(
            link for link in pages[filename]
            if link in pages
        )

    return pages


def transition_model(corpus, page, damping_factor):
    """
    Return a probability distribution over which page to visit next,
    given a current page.

    With probability `damping_factor`, choose a link at random
    linked to by `page`. With probability `1 - damping_factor`, choose
    a link at random chosen from all pages in the corpus.
    """
    pages = list(corpus.keys())
    n = len(pages)
    distribution = {}

    links = corpus[page]

    # Si no tiene links salientes, se comporta como si apuntara a todas
    if len(links) == 0:
        for p in pages:
            distribution[p] = 1 / n
        return distribution

    # Parte aleatoria: elegir cualquier página del corpus
    base_probability = (1 - damping_factor) / n
    for p in pages:
        distribution[p] = base_probability

    # Parte por links salientes
    linked_probability = damping_factor / len(links)
    for linked_page in links:
        distribution[linked_page] += linked_probability

    return distribution


def sample_pagerank(corpus, damping_factor, n):
    """
    Return PageRank values for each page by sampling `n` pages
    according to transition model, starting with a page at random.

    Return a dictionary where keys are page names, and values are
    their estimated PageRank value (a value between 0 and 1). All
    PageRank values should sum to 1.
    """
    counts = {page: 0 for page in corpus}

    # Primera muestra: página aleatoria
    current_page = random.choice(list(corpus.keys()))
    counts[current_page] += 1

    # Siguientes muestras según el modelo de transición
    for _ in range(n - 1):
        probabilities = transition_model(corpus, current_page, damping_factor)
        pages = list(probabilities.keys())
        weights = list(probabilities.values())

        current_page = random.choices(pages, weights=weights, k=1)[0]
        counts[current_page] += 1

    ranks = {}
    for page in counts:
        ranks[page] = counts[page] / n

    return ranks


def iterate_pagerank(corpus, damping_factor):
    """
    Return PageRank values for each page by iteratively updating
    PageRank values until convergence.

    Return a dictionary where keys are page names, and values are
    their estimated PageRank value (a value between 0 and 1). All
    PageRank values should sum to 1.
    """
    n = len(corpus)

    # Inicializar todos con 1/N
    ranks = {page: 1 / n for page in corpus}

    while True:
        new_ranks = {}

        for page in corpus:
            total = 0

            for possible_page in corpus:
                links = corpus[possible_page]

                # Si no tiene links, cuenta como si enlazara a todas
                if len(links) == 0:
                    total += ranks[possible_page] / n
                elif page in links:
                    total += ranks[possible_page] / len(links)

            new_ranks[page] = (1 - damping_factor) / n + damping_factor * total

        # Verificar convergencia
        converged = True
        for page in corpus:
            if abs(new_ranks[page] - ranks[page]) > 0.001:
                converged = False
                break

        ranks = new_ranks

        if converged:
            break

    # Normalizar para asegurar suma 1
    total_rank = sum(ranks.values())
    for page in ranks:
        ranks[page] /= total_rank

    return ranks


# En notebook, mejor deja esto comentado
# if __name__ == "__main__":
#     main()

In [3]:
corpus = {
    "1.html": {"2.html", "3.html"},
    "2.html": {"3.html"},
    "3.html": {"2.html"},
    "4.html": {"2.html", "3.html"}
}

print("Transition model desde 1.html:")
print(transition_model(corpus, "1.html", DAMPING))

print("\nSampling:")
print(sample_pagerank(corpus, DAMPING, 10000))

print("\nIteration:")
print(iterate_pagerank(corpus, DAMPING))

Transition model desde 1.html:
{'1.html': 0.037500000000000006, '2.html': 0.4625, '3.html': 0.4625, '4.html': 0.037500000000000006}

Sampling:
{'1.html': 0.039, '2.html': 0.4589, '3.html': 0.4634, '4.html': 0.0387}

Iteration:
{'1.html': 0.037500000000000006, '2.html': 0.4625, '3.html': 0.4625, '4.html': 0.037500000000000006}
